<a href="https://colab.research.google.com/github/ruchiaggarwalU/DS-AI-Auto-Telecom/blob/main/01-DS-LLM-Data-Analysis-Code/ds_loan_default_analysis_full_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Loan Default Prediction**


## **Problem Definition**

### **The Context:**

 - Why is this problem important to solve?

### **The objective:**

 - What is the intended goal?

### **The key questions:**

- What are the key questions that need to be answered?

### **The problem formulation**:

- What is it that we are trying to solve using data science?

## **Data Description:**
The Home Equity dataset (HMEQ) contains baseline and loan performance information for 5,960 recent home equity loans. The target (BAD) is a binary variable that indicates whether an applicant has ultimately defaulted or has been severely delinquent. This adverse outcome occurred in 1,189 cases (20 percent). 12 input variables were registered for each applicant.


* **BAD:** 1 = Client defaulted on loan, 0 = loan repaid

* **LOAN:** Amount of loan approved.

* **MORTDUE:** Amount due on the existing mortgage.

* **VALUE:** Current value of the property.

* **REASON:** Reason for the loan request. (HomeImp = home improvement, DebtCon= debt consolidation which means taking out a new loan to pay off other liabilities and consumer debts)

* **JOB:** The type of job that loan applicant has such as manager, self, etc.

* **YOJ:** Years at present job.

* **DEROG:** Number of major derogatory reports (which indicates a serious delinquency or late payments).

* **DELINQ:** Number of delinquent credit lines (a line of credit becomes delinquent when a borrower does not make the minimum required payments 30 to 60 days past the day on which the payments were due).

* **CLAGE:** Age of the oldest credit line in months.

* **NINQ:** Number of recent credit inquiries.

* **CLNO:** Number of existing credit lines.

* **DEBTINC:** Debt-to-income ratio (all your monthly debt payments divided by your gross monthly income. This number is one way lenders measure your ability to manage the monthly payments to repay the money you plan to borrow.

## <b>Important Notes</b>

- This notebook can be considered a guide to refer to while solving the problem. The evaluation will be as per the Rubric shared for the Milestone. Unlike previous courses, it does not follow the pattern of the graded questions in different sections. This notebook would give you a direction on what steps need to be taken in order to get a viable solution to the problem. Please note that this is just one way of doing this. There can be other 'creative' ways to solve the problem and we urge you to feel free and explore them as an 'optional' exercise.

- In the notebook, there are markdowns cells called - Observations and Insights. It is a good practice to provide observations and extract insights from the outputs.

- The naming convention for different variables can vary. Please consider the code provided in this notebook as a sample code.

- All the outputs in the notebook are just for reference and can be different if you follow a different approach.

- There are sections called **Think About It** in the notebook that will help you get a better understanding of the reasoning behind a particular technique/step. Interested learners can take alternative approaches if they want to explore different techniques.

### **Import the necessary libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score,precision_score,recall_score,f1_score

from sklearn import tree
from sklearn.tree import DecisionTreeClassifier

from sklearn import ensemble
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

import scipy.stats as stats

from sklearn.model_selection import GridSearchCV


import warnings
warnings.filterwarnings('ignore')

### **Read the dataset**

In [ ]:
#hm=pd.read_csv("hmeq.csv")

from google.colab import drive

# Read the data from CSV file
drive.mount('/content/drive', force_remount = True)
hm = pd.read_csv("/content/drive/MyDrive/CSV Files/Capstone_Data.csv")

In [ ]:
!jupyter nbconvert --to html /content/drive/MyDrive/Colab Notebooks/Capstone_Low_Code.ipynb

In [ ]:
# Copying data to another variable to avoid any changes to original data
data=hm.copy()

### **Print the first and last 5 rows of the dataset**

In [ ]:
data.head(20)

In [ ]:
# Display last 5 rows
data.tail(5)

### **Understand the shape of the dataset**

In [ ]:
# How many rows and columns in the data?
print("The data has", data.shape[0], "rows and", data.shape[1], "columns.")

**Insights: The data has 5960 rows and 13 columns.**

### **Check the data types of the columns**

In [ ]:
# What are the data types for the different columns?
data.info()

**Insights: Except REASON and JOB, all other fields are numeric.**

### **Check for missing values**

In [ ]:
# Check for duplicate values in the data
data.duplicated()

In [ ]:
# Print the statistical summary of the data
data.describe()

In [ ]:
# Print sorted list of count of missing values
data.isnull().sum().sort_values(ascending=False)

In [ ]:
# Check the percentage of missing values in each column
percent_missing = data.isnull().sum()*100/len(data)
missing_value_df = pd.DataFrame({'column name': data.columns,
                                 'percent_missing': percent_missing})

# Sort the percent missing values by column
missing_value_df.sort_values('percent_missing', inplace=True, ascending=False)
print(missing_value_df)

**Insights: Debt to Income ratio is missing in 21% of the records**

### **Think about it:**
- We found the total number of missing values and the percentage of missing values, which is better to consider?

It's better to consider the percentage of missing values rather than the absolute number of missing values, for the following reasons:

1. The total number of missing values can be misleading if you don't take into account the size of the dataset.

2. The percentage of missing values helps you decide whether to fill, drop, or handle the missing data differently.

3. When comparing missing values across different features, the percentage allows for a more meaningful comparison, especially when the features have different scales or number of entries.

- What can be the limit for % missing values in a column in order to avoid it and what are the challenges associated with filling them and avoiding them?

If a data set has > 10% missing values, better to not fill it because filling may impact the performance of the model.

**We can convert the object type columns to categories**

`converting "objects" to "category" reduces the data space required to store the dataframe`

### **Convert the data types**

In [ ]:
cols = data.select_dtypes(['object']).columns.tolist()

#adding target variable to this list as this is a classification problem and the target variable is categorical

cols.append('BAD')

In [ ]:
cols

In [ ]:
# Change the data type of object type column to category
for i in cols:
    data[i] = data[i].astype('category')

In [ ]:
# Check the info again and the datatype of different variable
data.info()

### **Analyze Summary Statistics of the dataset**

In [ ]:
# Analyze the summary statistics for numerical variables
data.describe().T

**Insights:
Median loan amount is \$16300.
Median mortgage value is \$65019.
Median home value is \$89235.**

In [ ]:
# Check summary for categorical data
data.describe(include=['category']).T

**Insights: Most loan applications are for debt consolidation.**

**Let's look at the unique values in all the categorical variables**

In [ ]:
# Check the count of unique values in each categorical column

cols_cat= data.select_dtypes(['category'])

for i in cols_cat.columns:
    print('Unique values in',i, ':')
    print(data[i].unique())
    print('*'*40)
    print()

**Insights:**

The absolute count of unique values in each categorical column is a useful metric, but it might not be sufficient on its own to assess the quality or importance of a categorical feature.

The absolute count of unique values tells you how many distinct categories are present, but it doesn’t inform you about the distribution of these categories.

Just knowing the number of unique categories doesn’t tell you how important the feature is for predicting the target variable.

A frequency distribution will be more useful.


### **Think about it**
- The results above gave the absolute count of unique values in each categorical column. Are absolute values a good measure?
- If not, what else can be used? Try implementing that.



In [ ]:
# Print frequency distribution of REASON
frequency_distribution = data['REASON'].value_counts(normalize=True)
print(frequency_distribution)

print()

# Print frequency distribution of JOB
frequency_distribution = data['JOB'].value_counts(normalize=True)
print(frequency_distribution)

## **Exploratory Data Analysis (EDA) and Visualization**

## **Univariate Analysis**

Univariate analysis is used to explore each variable in a data set, separately. It looks at the range of values, as well as the central tendency of the values. It can be done for both numerical and categorical variables

### **1. Univariate Analysis - Numerical Data**
Histograms and box plots help to visualize and describe numerical data. We use box plot and histogram to analyze the numerical columns.

In [ ]:
# While doing uni-variate analysis of numerical variables we want to study their central tendency and dispersion.
# Let us write a function that will help us create boxplot and histogram for any input numerical variable.
# This function takes the numerical column as the input and return the boxplots and histograms for the variable.
# Let us see if this help us write faster and cleaner code.
def histogram_boxplot(feature, figsize=(15,10), bins = None):
    """ Boxplot and histogram combined
    feature: 1-d feature array
    figsize: size of fig (default (9,8))
    bins: number of bins (default None / auto)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(nrows = 2, # Number of rows of the subplot grid= 2
                                           sharex = True, # x-axis will be shared among all subplots
                                           gridspec_kw = {"height_ratios": (.25, .75)},
                                           figsize = figsize
                                           ) # creating the 2 subplots
    sns.boxplot(feature, ax=ax_box2, showmeans=True, color='violet') # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(feature, kde=F, ax=ax_hist2, bins=bins,palette="winter") if bins else sns.histplot(feature, kde=False, ax=ax_hist2) # For histogram
    ax_hist2.axvline(np.mean(feature), color='green', linestyle='--') # Add mean to the histogram
    ax_hist2.axvline(np.median(feature), color='black', linestyle='-') # Add median to the histogram

#### Using the above function, let's first analyze the Histogram and Boxplot for LOAN

In [ ]:
# Build the histogram boxplot for Loan
histogram_boxplot(data['LOAN'])
print()

# Build the histogram boxplot for Mortgage Due
histogram_boxplot(data['MORTDUE'])

print()

# Build the histogram boxplot for Delinquencies
histogram_boxplot(data['DELINQ'])

**Insights __________**

#### **Note:** As done above, analyze Histogram and Boxplot for other variables

**Insights ____________**

### **2. Univariate Analysis - Categorical Data**

In [ ]:
# Function to create barplots that indicate percentage for each category.

def perc_on_bar(plot, feature):
    '''
    plot
    feature: categorical feature
    the function won't work if a column is passed in hue parameter
    '''

    total = len(feature) # length of the column
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height()/total) # percentage of each class of the category
        x = p.get_x() + p.get_width() / 2 - 0.05 # width of the plot
        y = p.get_y() + p.get_height()           # height of the plot
        ax.annotate(percentage, (x, y), size = 12) # annotate the percentage

    plt.show() # show the plot

#### Analyze Barplot for DELINQ

In [ ]:
#Build barplot for DELINQ
plt.figure(figsize=(15,5))
ax = sns.countplot(x=data["DELINQ"],palette='pastel')
perc_on_bar(ax,data["DELINQ"])

**Insights ________**

#### **Note:** As done above, analyze Histogram and Boxplot for other variables.

**Insights _____________**

## **Bivariate Analysis**

### **Bivariate Analysis: Continuous and Categorical Variables**

#### Analyze BAD vs Loan

**Insights ______**

In [ ]:
fig = plt.figure(figsize = (20,15))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

ax = fig.add_subplot(3, 3, 1)
sns.boxplot(x=data["BAD"],y=data['LOAN'],palette="PuBu")

ax = fig.add_subplot(3, 3, 2)
sns.boxplot(x=data["BAD"],y=data['VALUE'],palette="PuBu")

ax = fig.add_subplot(3, 3, 3)
sns.boxplot(x=data["BAD"],y=data['MORTDUE'],palette="PuBu")

ax = fig.add_subplot(3, 3, 4)
sns.boxplot(x=data["BAD"],y=data['DELINQ'],palette="PuBu")

ax = fig.add_subplot(3, 3, 5)
sns.boxplot(x=data["BAD"],y=data['CLNO'],palette="PuBu")

ax = fig.add_subplot(3, 3, 6)
sns.boxplot(x=data["BAD"],y=data['DEBTINC'],palette="PuBu")

ax = fig.add_subplot(3, 3, 7)
sns.boxplot(x=data["BAD"],y=data['CLNO'],palette="PuBu")

ax = fig.add_subplot(3, 3, 8)
sns.boxplot(x=data["BAD"],y=data['DEROG'],palette="PuBu")

ax = fig.add_subplot(3, 3, 9)
sns.boxplot(x=data["BAD"],y=data['NINQ'],palette="PuBu")

**Insights: DEROG and DELINQ have a definite impact on BAD. Followed by DEBTINC and NINQ.**

In [ ]:
### Function to plot stacked bar charts for categorical columns

def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
#    print(tab1)
#    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

def stacked_plot(x):
    sns.set(palette='nipy_spectral')
    tab1 = pd.crosstab(x,data['BAD'],margins=True)
    print(tab1)
    print('-'*120)
    tab = pd.crosstab(x,data['BAD'],normalize='index')
    tab.plot(kind='bar',stacked=True,figsize=(10,5))
    plt.legend(loc='lower left', frameon=False)
    plt.legend(loc="upper left", bbox_to_anchor=(1,1))
    plt.show()

### **Bivariate Analysis: Two Continuous Variables**

In [ ]:
fig = plt.figure(figsize = (20,15))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

ax = fig.add_subplot(3, 3, 1)
sns.scatterplot(x=data["VALUE"],y=data['LOAN'],palette="PuBu")

ax = fig.add_subplot(3, 3, 2)
sns.scatterplot(x=data["DELINQ"],y=data['LOAN'],palette="PuBu")

ax = fig.add_subplot(3, 3, 3)
sns.scatterplot(x=data["DELINQ"],y=data['MORTDUE'],palette="PuBu")

ax = fig.add_subplot(3, 3, 4)
sns.scatterplot(x=data["NINQ"],y=data['DELINQ'],palette="PuBu")

ax = fig.add_subplot(3, 3, 5)
sns.scatterplot(x=data["DEBTINC"],y=data['DELINQ'],palette="PuBu")

ax = fig.add_subplot(3, 3, 6)
sns.scatterplot(x=data["DEROG"],y=data['DEBTINC'],palette="PuBu")

ax = fig.add_subplot(3, 3, 7)
sns.scatterplot(x=data["DEROG"],y=data['CLNO'],palette="PuBu")

ax = fig.add_subplot(3, 3, 8)
sns.scatterplot(x=data["DEROG"],y=data['CLAGE'],palette="PuBu")

ax = fig.add_subplot(3, 3, 9)
sns.scatterplot(x=data["DEROG"],y=data['NINQ'],palette="PuBu")

### **Bivariate Analysis:  BAD vs Categorical Variables**

**The stacked bar chart (aka stacked bar graph)** extends the standard bar chart from looking at numeric values across one categorical variable to two.

#### Plot stacked bar plot for for BAD and REASON

In [ ]:
# Plot stacked bar plot for BAD and REASON
stacked_barplot(data, "REASON", "BAD")

**Insights: Loan Reason has an insignificant impact on default.**

#### **Note:** As shown above, perform Bivariate Analysis on different pairs of Categorical vs BAD

**Insights: No idea why the graph is appearing like this.**

### **Multivariate Analysis**

#### Analyze Correlation Heatmap for Numerical Variables

In [ ]:
# Separating numerical variables
numerical_col = data.select_dtypes(include=np.number).columns.tolist()

# Build correlation matrix for numerical columns
corr = data[numerical_col].corr()

# Plot the heatmap
plt.figure(figsize=(16,12))
sns.heatmap(corr,cmap='coolwarm',annot=True,linewidths=0.5,center=0,cbar=False,vmax=1,vmin=-1,
        fmt=".2f",
        xticklabels=corr.columns,
        yticklabels=corr.columns);

In [ ]:
# Build pairplot for the data with hue = 'BAD'
#numerical_data = data.select_dtypes(include=np.number)
numerical_data = data.drop(['REASON','JOB'],axis=1)

sns.pairplot(numerical_data, hue='BAD')
plt.show()

### **Think about it**
- Are there missing values and outliers in the dataset? If yes, how can you treat them?
- Can you think of different ways in which this can be done and when to treat these outliers or not?
- Can we create new features based on Missing values?

#### Treating Outliers

In [ ]:
def treat_outliers(df,col):
    '''
    treats outliers in a variable
    col: str, name of the numerical variable
    df: data frame
    col: name of the column
    '''

    Q1 = df[col].quantile(0.25)  # 25th quantile
    Q3 = df[col].quantile(0.75)  # 75th quantile
    IQR = Q3 - Q1                # IQR Range
    Lower_Whisker = Q1 - 1.5 * IQR  # define lower whisker
    Upper_Whisker = Q3 + 1.5 * IQR  # define upper whisker
    df[col] = np.clip(df[col], Lower_Whisker, Upper_Whisker) # all the values samller than Lower_Whisker will be assigned value of Lower_whisker
                                                            # and all the values above upper_whishker will be assigned value of upper_Whisker
    return df

def treat_outliers_all(df, col_list):
    '''
    treat outlier in all numerical variables
    col_list: list of numerical variables
    df: data frame
    '''
    for c in col_list:
        df = treat_outliers(df,c)

    return df


In [ ]:
df = data.copy()

df.info()
df.drop(['REASON','JOB'],axis=1,inplace=True)
df.info()

#### Adding new columns in the dataset for each column which has missing values

In [ ]:
#For each column we create a binary flag for the row, if there is missing value in the row, then 1 else 0.
def add_binary_flag(df,col):
    '''
    df: is the dataframe
    col: is column which has missing values
    Function returns a dataframe which has binary flag for missing values in column col
    '''
    new_col = str(col)
    new_col += '_missing_values_flag'
    df[new_col] = df[col].isna()
    return df

In [ ]:
# list of columns that has missing values in it
#missing_col = [col for col in df.columns if df[col].isnull().any()]

#for colmn in missing_col:
#    add_binary_flag(df,colmn)


#### Filling missing values in numerical columns with median and mode in categorical variables

In [ ]:
# Fill missing values in numerical columns with median, categorical columns with mode

# Select numeric columns.
num_data = df.select_dtypes('number')

# Select string and object columns.
cat_data = df.select_dtypes('category').columns.tolist()#df.select_dtypes('object')

# Fill numeric columns with median.
df[num_data.columns] = num_data.fillna(num_data.median())

# Fill object columns with mode.
for column in cat_data:
    mode = df[column].mode()[0]
    df[column] = df[column].fillna(mode)

## **Important Insights from EDA**

What are the the most important observations and insights from the data based on the EDA performed?

## **Model Building - Approach**
1. Data preparation
2. Partition the data into train and test set
3. Fit on the train data
4. Tune the model and prune the tree, if required
5. Test the model on test set

## **Data Preparation**

### **Separating the target variable from other variables**

In [ ]:
# Creating the copy of the dataframe
df1 = df.copy()

# Drop the dependent variable from the dataframe and create the X(independent variable) matrix
X = df1.drop(columns=['BAD'])

# Create dummy variables for the categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create y(dependent varibale)
y = df1['BAD']

### **Splitting the data into 70% train and 30% test set**

In [ ]:
# Split the data into training and test set
# Assuming we want to split the data with 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

### **Think about it**
- You can try different splits like 70:30 or 80:20 as per your choice. Does this change in split affect the performance?
- If the data is imbalanced, can you make the split more balanced and if yes, how?

## **Model Evaluation Criterion**

#### After understanding the problem statement, think about which evaluation metrics to consider and why.

In [ ]:
#@title
#creating metric function
def metrics_score(actual, predicted):
    print(classification_report(actual, predicted))
    cm = confusion_matrix(actual, predicted)
    plt.figure(figsize=(8,5))
    sns.heatmap(cm, annot=True,  fmt='.2f', xticklabels=['Not Eligible', 'Eligible'], yticklabels=['Not Eligible', 'Eligible'])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

### **Build a Logistic Regression Model**

In [ ]:
print(y_train.value_counts())

# Defining the Logistic regression model
model = LogisticRegression()

# Fitting the model on the training data
model.fit(X_train, y_train)

#### Checking the performance on the train dataset

In [ ]:
#Predict for train set
y_train_pred = model.predict(X_train)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Train Accuracy: {train_accuracy:.4f}")

# Generate classification report
train_classification_report = classification_report(y_train, y_train_pred)
print("Classification Report:\n", train_classification_report)

# Generate confusion matrix
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
print("Confusion Matrix:\n", train_confusion_matrix)

#### Checking the performance on the test dataset

In [ ]:
# Predict for test set
y_test_pred = model.predict(X_test)

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate classification report
test_classification_report = classification_report(y_test, y_test_pred)
print("Classification Report:\n", test_classification_report)

# Generate confusion matrix
test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:\n", test_confusion_matrix)

**Observations: __________**

#### Let's check the coefficients, and check which variables are important and how they affect the process of loan approval

In [ ]:
#@title
# Printing the coefficients of logistic regression
coefficients = model.coef_
intercept = model.intercept_

print("Coefficients of the model:", coefficients)
print("Intercept of the model:", intercept)

**Insights ________**

### **Think about it:**
- The above Logistic regression model was build on the threshold of 0.5, can we use different threshold?
- How to get an optimal threshold and which curve will help you achieve?
- How does, accuracy, precision and recall change on the threshold?

### **Build a Decision Tree Model**

### **Think about it:**
- In Logistic regression we treated the outliers and built the model, should we do the same for tree based models or not? If not, why?

For tree-based models like Decision Trees, Random Forests, the treatment of outliers is generally less critical. Tree-based models do not assume a specific distribution of the data. Also, the criteria for splitting in trees, such as information gain, are not significantly skewed by extreme values.

#### Data Preparation for the tree based model

In [ ]:
#@title
# Add binary flags
# List of columns that has missing values in it
#missing_col = [col for col in data.columns if data[col].isnull().any()]

#for colmn in missing_col:
#    add_binary_flag(data,colmn)


In [ ]:
#@title
#  Treat Missing values in numerical columns with median and mode in categorical variables
# Select numeric columns.
num_data = data.select_dtypes('number')

# Select string and object columns.
cat_data = data.select_dtypes('category').columns.tolist()#df.select_dtypes('object')

# Fill numeric columns with median.
# Remove _________ and complete the code
data[num_data.columns] = num_data.fillna(num_data.median())

# Fill object columns with model.
# Remove _________ and complete the code
for column in cat_data:
    mode = data[column].mode()[0]
    data[column] = data[column].fillna(mode)

#### Separating the target variable y and independent variable x

In [ ]:
# Creating the copy of the dataframe
df1 = df.copy()

# Drop the dependent variable from the dataframe and create the X(independent variable) matrix
X = df1.drop(columns=['BAD'])

# Create dummy variables for the categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create y(dependent varibale)
y = df1['BAD']

#### Split the data

In [ ]:
# Split the data into training and test set
# Assuming we want to split the data with 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
#@title
#Defining Decision tree model with class weights class_weight={0: 0.2, 1: 0.8}
model = DecisionTreeClassifier(class_weight={0: 0.2, 1: 0.8})

In [ ]:
#@title
#fitting Decision tree model
model.fit(X_train, y_train)

#### Checking the performance on the train dataset

In [ ]:
#@title
# Checking performance on the training data

# Make predictions on the training data
y_train_pred = model.predict(X_train)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Train Accuracy: {train_accuracy:.4f}")

# Generate classification report
train_classification_report = classification_report(y_train, y_train_pred)
print("Classification Report:\n", train_classification_report)

# Generate confusion matrix
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
print("Confusion Matrix:\n", train_confusion_matrix)

#### Checking the performance on the test dataset

In [ ]:
#@title
# Checking performance on the testing data

# Make predictions on the testing data
y_test_pred = model.predict(X_test)

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate classification report
test_classification_report = classification_report(y_test, y_test_pred)
print("Classification Report:\n", test_classification_report)

# Generate confusion matrix
test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:\n", test_confusion_matrix)

**Insights _____________**

### **Think about it:**
- Can we improve this model?
- How to get optimal parameters in order to get the best possible results?

### **Decision Tree - Hyperparameter Tuning**

* Hyperparameter tuning is tricky in the sense that **there is no direct way to calculate how a change in the hyperparameter value will reduce the loss of your model**, so we usually resort to experimentation. We'll use Grid search to perform hyperparameter tuning.
* **Grid search is a tuning technique that attempts to compute the optimum values of hyperparameters.**
* **It is an exhaustive search** that is performed on the specific parameter values of a model.
* The parameters of the estimator/model used to apply these methods are **optimized by cross-validated grid-search** over a parameter grid.

**Criterion {“gini”, “entropy”}**

The function to measure the quality of a split. Supported criteria are “gini” for the Gini impurity and “entropy” for the information gain.

**max_depth**

The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.

**min_samples_leaf**

The minimum number of samples is required to be at a leaf node. A split point at any depth will only be considered if it leaves at least min_samples_leaf training samples in each of the left and right branches. This may have the effect of smoothing the model, especially in regression.

You can learn about more Hyperpapameters on this link and try to tune them.

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html


#### Using GridSearchCV for Hyperparameter tuning on the model

In [ ]:
# Choose the type of classifier
d_tree_tuned = DecisionTreeClassifier(random_state = 14, class_weight = {0: 0.3, 1: 0.7})

# Grid of parameters to choose from
parameters = {'max_depth': np.arange(2, 10),
              'criterion': ['gini', 'entropy'],
              'min_samples_leaf': [5, 10, 20, 25]
             }

# Type of scoring used to compare parameter combinations - recall score for class 1
scorer = metrics.make_scorer(recall_score, pos_label = 1)

# Run the grid search
grid_obj = GridSearchCV(d_tree_tuned, parameters, scoring = scorer, cv = 5)

grid_obj = grid_obj.fit(X_train, y_train)

# Set the classifier to the best combination of parameters
d_tree_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data
d_tree_tuned = d_tree_tuned.fit(X_train, y_train)

#### Checking the performance on the train dataset

In [ ]:
# Check performance on the training data based on the tuned model
y_pred_train2 = d_tree_tuned.predict(X_train)

accuracy = metrics_score(y_train, y_pred_train2)

print(accuracy)

#### Checking the performance on the test dataset

In [ ]:
# Check performance on the test data based on the tuned model
y_pred_test2 = d_tree_tuned.predict(X_test)

accuracy = metrics_score(y_test, y_pred_test2)

print(accuracy)

**Insights ___________**

#### Plotting the Decision Tree

In [ ]:
# Draw the decision tree for the tuned model

features = list(X.columns)

plt.figure(figsize = (90, 30))

out = tree.plot_tree(d_tree_tuned, feature_names = features, impurity = False, filled = True, fontsize = 10, class_names = None)

for o in out:

    arrow = o.arrow_patch

    if arrow is not None:

        arrow.set_edgecolor('gray')

        arrow.set_linewidth(1)

plt.show()

INSIGHTS:

Following combination seems to have the lowest default rate:
* DELINQ < 0.5
* CLAGE < 178.2
* DEROG < 0.5
* LOAN < 18,100



In [ ]:
# Visualize and compare error on the training and test data for different values of max_depth in the decision tree classifier.
train_scores = []
test_scores = []

for depth in range(1, 10):

    d_tree3 = tree.DecisionTreeClassifier(criterion = 'entropy', max_depth = depth)

    d_tree3 = d_tree3.fit(X_train, y_train)

    train_scores.append(1 - d_tree3.score(X_train, y_train))

    test_scores.append(1 - d_tree3.score(X_test, y_test))

plt.plot(range(1, 10), train_scores, '-o', label = "train")

plt.plot(range(1, 10), test_scores, '-o', label = "test")

plt.legend(loc = 'best')

plt.xlabel('max depth')

plt.ylabel('misclassification error (%)')

# Setting the range of the Y-axis
plt.ylim(0, 0.3)

plt.title("Decision Tree")

fig = plt.gcf()

fig.set_size_inches(10, 6)

plt.show()

### **Building a Random Forest Classifier**

**Random Forest is a bagging algorithm where the base models are Decision Trees.** Samples are taken from the training data and on each sample a decision tree makes a prediction.

**The results from all the decision trees are combined together and the final prediction is made using voting or averaging.**

In [ ]:
# Fitting the random forest tree classifier on the training data
rf_estimator =  ensemble.RandomForestClassifier()

rf_estimator = rf_estimator.fit(X_train, y_train)

#### Checking the performance on the train dataset

In [ ]:
# Checking performance on the training data
y_pred_train3 = rf_estimator.predict(X_train)

#### Checking the performance on the test dataset

In [ ]:
#@title
# Checking performance on the test data
y_pred_test3 = rf_estimator.predict(X_test)

**Observations: __________**

### **Build a Random Forest model with Class Weights**

In [ ]:
# Choose the type of classifier
rf_estimator_tuned = RandomForestClassifier(criterion = "entropy", random_state = 7)

# Grid of parameters to choose from
parameters = {"n_estimators": [110, 120],
    "max_depth": [6, 7],
    "min_samples_leaf": [20, 25],
    "max_features": [0.8, 0.9],
    "max_samples": [0.9, 1],
    "class_weight": ["balanced",{0: 0.3, 1: 0.7}]
}

# Type of scoring used to compare parameter combinations - recall score for class 1
scorer = metrics.make_scorer(recall_score, pos_label = 1)

# Run the grid search on the training data using scorer=scorer and cv=5
grid_obj = GridSearchCV(rf_estimator_tuned, parameters, scoring = scorer, cv = 5)

grid_obj = grid_obj.fit(X_train, y_pred_train3)

# Save the best estimator to variable rf_estimator_tuned
rf_estimator_tuned = grid_obj.best_estimator_

# Fit the best estimator to the training data
rf_estimator_tuned = rf_estimator_tuned.fit(X_train, y_pred_train3)

#### Checking the performance on the train dataset

In [ ]:
# Checking performance on the train data
y_pred_train4 = rf_estimator_tuned.predict(X_train)

#### Checking the performance on the test dataset

In [ ]:
#@title
# Checking performance on the test data
y_pred_test4 = rf_estimator_tuned.predict(X_test)

### **Think about it:**
- Can we try different weights?
- If yes, should we increase or decrease class weights for different classes?

### **Tuning the Random Forest**

* Hyperparameter tuning is tricky in the sense that **there is no direct way to calculate how a change in the hyperparameter value will reduce the loss of your model**, so we usually resort to experimentation. We'll use Grid search to perform hyperparameter tuning.
* **Grid search is a tuning technique that attempts to compute the optimum values of hyperparameters.**
* **It is an exhaustive search** that is performed on the specific parameter values of a model.
* The parameters of the estimator/model used to apply these methods are **optimized by cross-validated grid-search** over a parameter grid.


**n_estimators**: The number of trees in the forest.

**min_samples_split**: The minimum number of samples required to split an internal node:

**min_samples_leaf**: The minimum number of samples required to be at a leaf node.

**max_features{“auto”, “sqrt”, “log2”, 'None'}**: The number of features to consider when looking for the best split.

- If “auto”, then max_features=sqrt(n_features).

- If “sqrt”, then max_features=sqrt(n_features) (same as “auto”).

- If “log2”, then max_features=log2(n_features).

- If None, then max_features=n_features.

You can learn more about Random Forest Hyperparameters from the link given below and try to tune them

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

#### **Warning:** This may take a long time depending on the parameters you tune.

In [ ]:
#@title
# Choose the type of classifier
model = RandomForestClassifier()

# Grid of parameters to choose from
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Type of scoring used to compare parameter combinations
scoring = 'accuracy'

# Run the grid search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=5, n_jobs=-1, verbose=2)

# Fit the GridSearch on train dataset
grid_search.fit(X_train, y_train)

# Set the model to the best combination of parameters
best_model = grid_search.best_estimator_

# Fit the best algorithm to the data
best_model.fit(X_train, y_train)

#### Checking the performance on the train dataset

In [ ]:
#@title
# Checking performance on the train data
y_pred_train4 = best_model.predict(X_train)

#### Checking the performance on the test dataset

In [ ]:
#@title
# Checking performace on test dataset
y_pred_test4 = best_model.predict(X_test)

**Insights: _____**

In [ ]:
# Visualize and compare error on the training and test data for different values of max_depth in the random forest classifier.
train_scores = []
test_scores = []

for depth in range(1, 10):

    rf1 = RandomForestClassifier(random_state = 0, criterion = 'entropy', max_depth = depth, n_estimators = 25)

    rf1 = rf1.fit(X_train, y_train)

    train_scores.append(1 - rf1.score(X_train, y_train))

    test_scores.append(1 - rf1.score(X_test, y_test))

plt.plot(range(1, 10), train_scores, '-o', label = "train")

plt.plot(range(1, 10), test_scores, '-o', label = "test")

plt.legend(loc = 'upper right')

plt.xlabel('max depth')

plt.ylim(0, 0.3)

plt.ylabel('misclassification error (%)')

plt.title(f"Random Forest with 25 estimators")

fig = plt.gcf()

fig.set_size_inches(10, 6)

plt.show()

**Insights: Random Forest misclassification error rate looks worse than the one for Decision Tree.**

#### Plot the Feature importance of the tuned Random Forest

In [ ]:
#@title
# importance of features in the tree building ( The importance of a feature is computed as the
#(normalized) total reduction of the criterion brought by that feature. It is also known as the Gini importance )
# Checking performace on test dataset
# Remove _________ and complete the code

importances = best_model.feature_importances_

indices = np.argsort(importances)

feature_names = list(X.columns)

plt.figure(figsize = (12, 12))

plt.title('Feature Importances')

plt.barh(range(len(indices)), importances[indices], color = 'gray', align = 'center')

plt.yticks(range(len(indices)), [feature_names[i] for i in indices])

plt.xlabel('Relative Importance')

plt.show()

**Insights: Above chart shows relative importance of factors affecting loan default rate. Debt-to-Income ratio is most impactful, and Reason and Job type were least impactful so they have been removed from the data set.**

### **Think about it:**
- We have only built 3 models so far, Logistic Regression, Decision Tree and Random Forest
- We can build other Machine Learning classification models like kNN, LDA, QDA or even Support Vector Machines (SVM).
- Can we also perform feature engineering and create model features and build a more robust and accurate model for this problem statement?

### **Comparing Model Performances**

In [ ]:
#@title
def get_recall_score(model,flag=True,X_train=X_train,X_test=X_test):
    '''
    model : classifier to predict values of X

    '''
    a = [] # defining an empty list to store train and test results
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_recall = metrics.recall_score(y_train,pred_train)
    test_recall = metrics.recall_score(y_test,pred_test)
    a.append(train_recall) # adding train recall to list
    a.append(test_recall) # adding test recall to list

    # If the flag is set to True then only the following print statements will be dispayed. The default value is set to True.
    if flag == True:
        print("Recall on training set : ",metrics.recall_score(y_train,pred_train))
        print("Recall on test set : ",metrics.recall_score(y_test,pred_test))

    return a # returning the list with train and test scores

In [ ]:
#@title
##  Function to calculate precision score
def get_precision_score(model,flag=True,X_train=X_train,X_test=X_test):
    '''
    model : classifier to predict values of X

    '''
    b = []  # defining an empty list to store train and test results
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_precision = metrics.precision_score(y_train,pred_train)
    test_precision = metrics.precision_score(y_test,pred_test)
    b.append(train_precision) # adding train precision to list
    b.append(test_precision) # adding test precision to list

    # If the flag is set to True then only the following print statements will be dispayed. The default value is set to True.
    if flag == True:
        print("Precision on training set : ",metrics.precision_score(y_train,pred_train))
        print("Precision on test set : ",metrics.precision_score(y_test,pred_test))

    return b # returning the list with train and test scores

In [ ]:
#@title
##  Function to calculate accuracy score
def get_accuracy_score(model,flag=True,X_train=X_train,X_test=X_test):
    '''
    model : classifier to predict values of X

    '''
    c = [] # defining an empty list to store train and test results
    train_acc = model.score(X_train,y_train)
    test_acc = model.score(X_test,y_test)
    c.append(train_acc) # adding train accuracy to list
    c.append(test_acc) # adding test accuracy to list

    # If the flag is set to True then only the following print statements will be dispayed. The default value is set to True.
    if flag == True:
        print("Accuracy on training set : ",model.score(X_train,y_train))
        print("Accuracy on test set : ",model.score(X_test,y_test))

    return c # returning the list with train and test scores

In [ ]:
#@title
# Make the list of all the model names

models = [best_model]

# defining empty lists to add train and test results
acc_train = []
acc_test = []
recall_train = []
recall_test = []
precision_train = []
precision_test = []

# looping through all the models to get the accuracy,recall and precision scores
for model in models:
     # accuracy score
    j = get_accuracy_score(model,False)
    acc_train.append(j[0])
    acc_test.append(j[1])

    # recall score
    k = get_recall_score(model,False)
    recall_train.append(k[0])
    recall_test.append(k[1])

    # precision score
    l = get_precision_score(model,False)
    precision_train.append(l[0])
    precision_test.append(l[1])

In [ ]:
#@title
# Mention the Model names in the list. for example 'Model': ['Decision Tree', 'Tuned Decision Tree'..... write names of all models built]

comparison_frame = pd.DataFrame({'Model':['Decision Tree'],
                                          'Train_Accuracy': acc_train,
                                          'Test_Accuracy': acc_test,
                                          'Train_Recall': recall_train,
                                          'Test_Recall': recall_test,
                                          'Train_Precision': precision_train,
                                          'Test_Precision': precision_test})
comparison_frame

**Insights: ________**

**1. Comparison of various techniques and their relative performance based on chosen Metric (Measure of success):**
- How do different techniques perform? Which one is performing relatively better? Is there scope to improve the performance further?

**2. Refined insights:**
- What are the most meaningful insights relevant to the problem?

Debt-to-Income ratio is most impactful, and Reason and Job type were least impactful so they have been removed from the data set.

**3. Proposal for the final solution design:**
- What model do you propose to be adopted? Why is this the best solution to adopt?

**Random Forest** would be a good solution, for it's not clear whether it's working in the above calculations. Looking at the misclassification error rate, it seems Decision Trees are performing better w.r.t. accuracy. Random Forests can be computationally expensive and less interpretable than simpler models.

**Gradient Boosting Machines** (XGBoost, LightGBM) can also be used due to their balance between performance and flexibility, especially in dealing with complex, imbalanced datasets.